In [1]:
import pymupdf4llm

# Конвертируем весь PDF в один Markdown-строку
md_text = pymupdf4llm.to_markdown("Mois.pdf")

In [2]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "Header_1"),
    ("##", "Header_2"),
    ("###", "Header_3"),
]

splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
sections = splitter.split_text(md_text)

In [9]:
import json
import time
import uuid
from openai import OpenAI

client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

# Обновленный системный промпт
SYSTEM_PROMPT = """Ты — ведущий инженер знаний и эксперт по математической логике. Твоя задача — трансформировать учебный текст в строгую пропозициональную базу знаний.

КРИТИЧЕСКИЕ ПРАВИЛА:
1. ИГНОРИРУЙ служебную информацию (оглавления, литературу, лабораторные работы, тестовые главы). Бери только теоретические сведения.
2. ВЫДЕЛЯЙ атомарные факты: одно утверждение — один объект.
3. ПРИМЕРЫ: Определение (natural_language) должно быть строгим и абстрактным. Все конкретные примеры из текста (с цифрами, буквами, объектами) сохраняй СТРОГО в массив "examples".
4. ТИПИЗАЦИЯ: строго одно из [definition, axiom, theorem, constraint].
5. КОНФЛИКТЫ: поле "conflicts_with" должно содержать прямое логически противоположное утверждение(с возможностью использовать это для теста).
6. ФОРМУЛЫ: в поле "formal_logic" используй LaTeX.
7. В DOMAIN выбирай строго одно из [Теория Множеств, Теория формальных языков, Общая теория интеллектуальных систем, Технология OSTIS(Всё, что связано с sc, scg, scs, scn, scp)]

Формат вывода — СТРОГИЙ массив JSON без markdown-разметки вокруг. Если полезных фактов нет, верни пустой массив []."""

def process_to_propositional_json(sections):
    knowledge_base = []
    
    for i, section in enumerate(sections):
        headers = [v for k, v in section.metadata.items() if "Header" in k]
        current_title = headers[-1] if headers else "Без заголовка"
        domain = headers[0] if headers else "Общая теория"

        # Обновленный шаблон JSON
        prompt = f"""Проанализируй текст из раздела '{current_title}' и извлеки атомарные факты в виде массива JSON.

ТЕКСТ ДЛЯ АНАЛИЗА:
{section.page_content}

ОЖИДАЕМЫЙ ФОРМАТ ВЫВОДА:
[
  {{
    "id": "Сгенерируй ID",
    "entity_name": "Чистое название концепта",
    "domain": "Теория Множеств | Теория формальных языков | Общая теория интеллектуальных систем | Технология OSTIS(Всё, что связано с sc, scg, scs, scn, scp)",
    "type": "definition | axiom | theorem | constraint",
    "content": {{
      "natural_language": "Точная формулировка факта (без примеров)",
      "formal_logic": "Формула в LaTeX или логика предикатов",
      "explanation": "Краткое пояснение сути",
      "examples": ["Пример 1 из текста (если есть)", "Пример 2 из текста"]
    }},
    "relations": {{
      "parent_id": "Названание понятия(concept) являющегося надклассом для данного(если есть)",
      "depends_on": ["Названия базовых концептов, используемых в точной формулировке факта"],
      "conflicts_with": ["Логически противоположное утверждение для тестирования галлюцинаций на естественном языке"]
    }}
  }}
]"""

        try:
            response = client.chat.completions.create(
                model="qwen2.5-coder-7b-instruct",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.1,
                max_tokens=4096, 
                timeout=60
            )
            
            raw_res = response.choices[0].message.content.strip()
            
            if raw_res.startswith("```"):
                raw_res = raw_res.split("\n", 1)[1]
            if raw_res.endswith("```"):
                raw_res = raw_res.rsplit("\n", 1)[0]
                
            section_data = json.loads(raw_res)
            
            if isinstance(section_data, list) and len(section_data) > 0:
                for item in section_data:
                    item["id"] = f"FACT_{uuid.uuid4().hex[:8].upper()}"
                    knowledge_base.append(item)
                print(f"[{i+1}/{len(sections)}] + Извлечено фактов: {len(section_data)} из {current_title}")
            else:
                print(f"[{i+1}/{len(sections)}] - Пропущен мусорный раздел: {current_title}")
                
        except json.JSONDecodeError:
            print(f"[{i+1}/{len(sections)}] ! Ошибка JSON: Модель вернула невалидный формат.")
        except Exception as e:
            print(f"[{i+1}/{len(sections)}] ! Ошибка в секции: {e}")
            
    return knowledge_base

tree_data = process_to_propositional_json(sections[8:250])

with open('propositional_kb_1.json', 'w', encoding='utf-8') as f:
    json.dump(tree_data, f, ensure_ascii=False, indent=4)

[1/242] + Извлечено фактов: 2 из **1.1.3. Конечные и бесконечные множества**
[2/242] + Извлечено фактов: 1 из **1.1.3.1. Понятие конечного множества**
[3/242] + Извлечено фактов: 1 из **1.1.3.2. Понятие бесконечного множества**
[4/242] + Извлечено фактов: 1 из **1.1.4. Понятие пустого множества**
[5/242] + Извлечено фактов: 1 из **1.1.5.1. Понятие подмножества**
[6/242] + Извлечено фактов: 1 из **1.1.5.2. Понятие надмножества**
[7/242] + Извлечено фактов: 1 из **1.1.6. Операции над множествами**
[8/242] + Извлечено фактов: 1 из **1.1.6.1. Понятие объединения множеств**
[9/242] + Извлечено фактов: 3 из **1.1.6.2. Понятие пересечения множеств**
[10/242] + Извлечено фактов: 1 из **1.1.6.3. Понятие разности двух множеств**
[11/242] + Извлечено фактов: 1 из **1.1.6.4. Понятие симметрической разности двух множеств**
[12/242] + Извлечено фактов: 1 из **1.1.6.5. Понятие булеана множества**
[13/242] + Извлечено фактов: 1 из **1.2. Понятие связки**
[14/242] + Извлечено фактов: 1 из **1.2.1. Поня

In [1]:
import json
import faiss
import numpy as np
import os
import torch
import pickle
import re
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

INPUT_KB = 'propositional_kb.json'
MODEL_PATH = './hugg'
FAISS_OUT = "tree_kb.faiss"
METADATA_OUT = "tree_metadata.json"
BM25_OUT = "tree_kb_bm25.pkl" # Новый файл для сохранения BM25 индекса

def tokenize_ru(text):
    # Простая токенизация: приводим к нижнему регистру и оставляем только слова
    return re.findall(r'\w+', text.lower())

def build_vector_db_from_propositional_kb(kb_file):
    if not os.path.exists(kb_file):
        print(f"Ошибка: Файл {kb_file} не найден!")
        return

    with open(kb_file, 'r', encoding='utf-8') as f:
        kb_data = json.load(f)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"--- Загрузка модели из {MODEL_PATH} на {device} ---")
    model = SentenceTransformer(MODEL_PATH, device=device)

    documents = []
    metadata_list = []
    tokenized_corpus = [] # Для BM25
    
    print("--- Анализ пропозициональной базы знаний ---")
    
    for fact in kb_data:
        entity = fact.get("entity_name", "Неизвестно")
        domain = fact.get("domain", "Общая область")
        content = fact.get("content", {})
        relations = fact.get("relations", {})
        
        natural_text = content.get("natural_language", "")
        logic_formula = content.get("formal_logic", "")
        explanation = content.get("explanation", "")
        examples = content.get("examples", [])
        conflicts = relations.get("conflicts_with", [])
        
        # 1. ТЕКСТ ДЛЯ ЭМБЕДДИНГА (Entity Boosting)
        # Жестко фиксируем фокус на термине
        embed_parts = [f"Термин: {entity}. Определение {entity}: {natural_text}"]
        
        if explanation:
            embed_parts.append(f"Иными словами: {explanation}")
        if examples:
            # Добавляем примеры, чтобы FAISS "видел" их
            embed_parts.append(f"Примеры {entity}: {'; '.join(examples)}")
        if conflicts:
            # Добавляем анти-факты
            embed_parts.append(f"Это противоречит: {'; '.join(conflicts)}")
            
        embed_text = " ".join(embed_parts)
        documents.append(embed_text)
        
        # Токенизируем для BM25 (используем тот же текст)
        tokenized_corpus.append(tokenize_ru(embed_text))
        
        # 2. ПОЛНЫЙ КОНТЕКСТ ДЛЯ NLI (Rich Context)
        rich_context = f"Термин: {entity} (Область: {domain}). Определение: {natural_text} "
        if logic_formula: rich_context += f"Формально: {logic_formula}. "
        if explanation: rich_context += f"Пояснение: {explanation}. "
        if examples: rich_context += f"Примеры: {'; '.join(examples)}. "
        if conflicts: rich_context += f"Логические анти-факты: {'; '.join(conflicts)}."
        
        metadata_list.append({
            "term": entity,
            "domain": domain,
            "rich_context": rich_context,
            "formal_logic": logic_formula,
            "conflicts": conflicts
        })

    print(f"--- Создание индекса BM25 ---")
    bm25 = BM25Okapi(tokenized_corpus)
    with open(BM25_OUT, "wb") as f:
        pickle.dump(bm25, f)

    print(f"--- Векторизация {len(documents)} объектов (FAISS) ---")
    if torch.cuda.is_available(): torch.cuda.empty_cache()
        
    embeddings = model.encode(documents, show_progress_bar=True, batch_size=32)
    faiss.normalize_L2(embeddings)
    embeddings = np.array(embeddings).astype('float32')

    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension) 
    index.add(embeddings)
    faiss.write_index(index, FAISS_OUT)
    
    with open(METADATA_OUT, "w", encoding="utf-16") as f:
        json.dump(metadata_list, f, ensure_ascii=False, indent=4)
    
    print(f"Успех! Созданы файлы: {FAISS_OUT}, {METADATA_OUT}, {BM25_OUT}")

if __name__ == "__main__":
    build_vector_db_from_propositional_kb(INPUT_KB)

--- Загрузка модели из ./hugg на cuda ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

--- Анализ пропозициональной базы знаний ---
--- Создание индекса BM25 ---
--- Векторизация 543 объектов (FAISS) ---


Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Успех! Созданы файлы: tree_kb.faiss, tree_metadata.json, tree_kb_bm25.pkl


In [3]:
import json
import faiss
import torch
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("--- Загрузка моделей ---")
search_model = SentenceTransformer('./hugg')

nli_model_path = './bert'
tokenizer = AutoTokenizer.from_pretrained(nli_model_path)
nli_model = AutoModelForSequenceClassification.from_pretrained(nli_model_path)

index = faiss.read_index("tree_kb.faiss")
with open("tree_metadata.json", "r", encoding="utf-16") as f:
    metadata = json.load(f)

def check_hallucination(claim):
    # --- ШАГ 1: Поиск в векторной базе FAISS ---
    # Векторизуем утверждение пользователя
    query_vec = search_model.encode([claim]).astype('float32')
    faiss.normalize_L2(query_vec)
    
    # Ищем топ-3 наиболее похожих фрагмента из базы знаний
    k_matches = 3
    distances, indices = index.search(query_vec, k=k_matches)
    
    # Устанавливаем порог сходства (Threshold)
    # Если сходство ниже 0.86, считаем, что информации в базе нет
    SIMILARITY_THRESHOLD = 0.86 
    
    print(f"\n[Проверяемое утверждение]: {claim}")
    
    if distances[0][0] < SIMILARITY_THRESHOLD:
        print(f"--- Результат проверки ---")
        print(f"СТАТУС: Нет данных (Максимальное сходство: {distances[0][0]:.2f})")
        print("В базе знаний не найдено релевантной информации для проверки этого факта.")
        return "Нет данных"

    # Загружаем маппинг лейблов из конфигурации NLI-модели
    id2label = nli_model.config.id2label

    best_entail_prob = -1
    best_result_dict = None
    best_match_term = ""

    print("[Логический анализ найденных совпадений]:")
    
    # --- ШАГ 2: Поочередная проверка каждого найденного совпадения ---
    for i in range(k_matches):
        sim = distances[0][i]
        
        # Анализируем только те результаты, которые близки к лучшему совпадению
        if sim >= (SIMILARITY_THRESHOLD - 0.05):
            idx = indices[0][i]
            match = metadata[idx]
            
            # Извлекаем термин и его определение
            term = match.get('term', '').lower()
            context = match.get('rich_context', '') 
            
            # Приводим текст к естественному виду для лучшего понимания моделью
            natural_text = context.replace("Термин: ", "").replace(". Определение: ", " — это ")
            
            # --- СЕМАНТИЧЕСКИЙ ФИЛЬТР (Subject Match) ---
            # Проверяем, упоминается ли найденный термин в проверяемом утверждении.
            # Это защищает от ложных подтверждений похожих, но разных понятий.
            claim_lower = claim.lower()
            subject_match = term in claim_lower or any(word in claim_lower for word in term.split() if len(word) > 3)
            
            # Подаем пару (Эталон, Утверждение) в NLI-модель
            inputs = tokenizer(natural_text, claim, truncation=True, max_length=512, return_tensors="pt")
            with torch.no_grad():
                outputs = nli_model(**inputs)
                probs = torch.softmax(outputs.logits, dim=1).tolist()[0]
            
            current_results = {}
            current_entail = 0
            neutral_val = 0  # Добавляем переменную для отслеживания нейтральности
            
            for j, prob in enumerate(probs):
                label_name = id2label[j].lower()
                val = prob
                
                if 'entail' in label_name:
                    if not subject_match:
                        val *= 0.5
                    current_entail = val
                    current_results['Подтверждено'] = val
                elif 'contradiction' in label_name:
                    current_results['Противоречие (Галлюцинация)'] = val
                else:
                    neutral_val = val # Сохраняем значение нейтральности
                    current_results['Нейтрально'] = val

            # --- НОВОЕ УСЛОВИЕ: Подавление при доминировании Нейтральности ---
            # Если Нейтрально > 80% и оно значительно выше Подтверждения
            if neutral_val > 0.80 and neutral_val > current_entail:
                 print(f"  -> '{term}' | Слишком высокая неопределенность ({neutral_val*100:.1f}%) - игнорируем.")
                 continue # Пропускаем этот результат, он не дает четкого ответа
            
            status_msg = "(Subject Match!)" if subject_match else "(Wrong Subject - Penalty applied)"
            print(f"  -> '{term}' | Подтверждение: {current_entail*100:.1f}% {status_msg}")
            
            # Выбираем тот фрагмент базы, который дает наиболее уверенное подтверждение
            if current_entail > best_entail_prob:
                best_entail_prob = current_entail
                best_result_dict = current_results
                best_match_term = term

    # --- ШАГ 3: Формирование итогового вердикта ---
    print(f"\n--- Итоговый результат (на основе '{best_match_term}') ---")
    
    if best_result_dict is None:
        return "Нет данных"

    # Сортируем результаты по вероятности для вывода
    sorted_res = sorted(best_result_dict.items(), key=lambda x: x[1], reverse=True)
    for label, prob in sorted_res:
        print(f"{label}: {prob*100:.2f}%")
    
    return max(best_result_dict, key=best_result_dict.get)

# Можешь запустить свои тесты здесь

# --- ТЕСТЫ ---
print("\n--- Тест 1: Правильное утверждение ---")
check_hallucination("Неориентированная связка связка, в которой элементы имеют одинаковые роли или не имеют их вовсе.")

print("\n--- Тест 2: Галлюцинация ---")
check_hallucination("Бинарное отношение — это множество пар, где первый элемент из одного множества, а второй — из другого.")

print("\n--- Тест 3: Правильное утверждение (Синонимы) ---")
# Проверяем определение ИС из раздела 1
check_hallucination("Интеллектуальная система — это программный комплекс, способный справляться с интеллектуальными задачами.")

print("\n--- Тест 4: Галлюцинация (Прямое противоречие) ---")
# Проверяем против факта: "Любой формальный язык задаётся при помощи языка математики"
check_hallucination("Математический язык не используется для описания формальных языков.")

print("\n--- Тест 5: Правильное утверждение (Атомарный факт) ---")
# Проверяем информацию о Лабораторной работе №4 из оглавления
check_hallucination("Лабораторная работа №4 посвящена разработке программ на языке асинхронного процедурного программирования SCP.")

print("\n--- Тест 6: Галлюцинация (Подмена понятий) ---")
# Проверяем против факта: "Мощность множества — это число элементов этого множества"
check_hallucination("Мощность множества — это название сущностей, из которых состоит данное множество.")

print("\n--- Тест 7: Правильное утверждение (Сложное определение) ---")
# Проверяем определение конечного множества из раздела 1.1.3.1
check_hallucination("Конечное множество — это такое множество, мощность которого равна некоторому неотрицательному целому числу.")

--- Загрузка моделей ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]


--- Тест 1: Правильное утверждение ---

[Проверяемое утверждение]: Неориентированная связка связка, в которой элементы имеют одинаковые роли или не имеют их вовсе.
[Логический анализ найденных совпадений]:
  -> 'неориентированная связка' | Подтверждение: 98.8% (Subject Match!)
  -> 'неориентированная связка' | Подтверждение: 4.0% (Subject Match!)
  -> 'ориентированная связка' | Слишком высокая неопределенность (83.7%) - игнорируем.

--- Итоговый результат (на основе 'неориентированная связка') ---
Подтверждено: 98.78%
Нейтрально: 0.98%
Противоречие (Галлюцинация): 0.27%

--- Тест 2: Галлюцинация ---

[Проверяемое утверждение]: Бинарное отношение — это множество пар, где первый элемент из одного множества, а второй — из другого.
[Логический анализ найденных совпадений]:
  -> 'бинарное отношение' | Подтверждение: 4.6% (Subject Match!)
  -> 'бинарное отношение' | Подтверждение: 21.4% (Subject Match!)
  -> 'отношение' | Подтверждение: 3.0% (Subject Match!)

--- Итоговый результат (на осно

'Подтверждено'